Copyright **`(c)`** 2025 Giovanni Squillero `<giovanni.squillero@polito.it>`  
[`https://github.com/squillero/computational-intelligence`](https://github.com/squillero/computational-intelligence)  
Free under certain conditions — see the [`license`](https://github.com/squillero/computational-intelligence/blob/master/LICENSE.md) for details.  

In [2]:
from itertools import product, combinations
import numpy as np
import networkx as nx
from icecream import ic

In [3]:
def create_problem(
    size: int,
    *,
    density: float = 1.0,
    negative_values: bool = False,
    noise_level: float = 0.0,
    seed: int = 42,
) -> np.ndarray:
    """Problem generator for Lab3"""
    rng = np.random.default_rng(seed)
    # Genera N (size) punti casuali nel piano
    map = rng.random(size=(size, 2))
    # Inizializza la matrice dei pesi casuali
    problem = rng.random((size, size))
    if negative_values:
        problem = problem * 2 - 1
    problem *= noise_level
    for a, b in product(range(size), repeat=2):
        if rng.random() < density:
            # se l'arco esiste distanza euclidea + rumore (dell'inizializzazione)
            problem[a, b] += np.sqrt(
                np.square(map[a, 0] - map[b, 0]) + np.square(map[a, 1] - map[b, 1])
            )
        else:
            # se l'arco non esiste peso infinito
            problem[a, b] = np.inf
    np.fill_diagonal(problem, 0)
    return (problem * 1_000).round()


# Crea N punti in un piano 2D.
#Calcola le loro distanze euclidee.
#Usa queste distanze come pesi degli archi del grafo.
#Con density < 1, alcuni archi vengono rimossi (posti a inf → nessun collegamento).

#Aggiunge eventualmente rumore e valori negativi.

In [75]:
problem = create_problem(20, density=0.15, noise_level=10, negative_values=True)
problem

array([[ 0.000e+00,        inf,        inf,        inf,        inf,
               inf,        inf,        inf,  4.084e+03,        inf,
        -5.916e+03,        inf,        inf,        inf,  4.769e+03,
               inf,        inf,        inf,        inf,        inf],
       [       inf,  0.000e+00,        inf,        inf,        inf,
               inf,        inf,        inf, -8.681e+03,        inf,
               inf,        inf,  7.577e+03, -4.559e+03,        inf,
               inf,        inf,        inf,        inf,        inf],
       [       inf,        inf,  0.000e+00,        inf, -9.019e+03,
               inf,        inf,        inf,        inf,        inf,
               inf,        inf,        inf,        inf,        inf,
               inf,        inf,        inf,        inf,        inf],
       [       inf,        inf,        inf,  0.000e+00,        inf,
               inf,        inf,        inf,        inf,  8.220e+03,
        -4.530e+02,        inf, -3.289e+03,  

In [86]:
masked = np.ma.masked_array(problem, mask=np.isinf(problem))
G = nx.from_numpy_array(masked, create_using=nx.DiGraph)

In [88]:
for s, d in combinations(range(problem.shape[0]), 2):
    try:
        # path = nx.shortest_path(G, s, d, weight='weight')
        path = nx.bellman_ford_path(G, s, d, weight='weight')
        #cost = cost = nx.path_weight(G, path, weight='weight')
    except nx.NetworkXNoPath:
        # Nodes are not connected
        path = None
        cost = np.inf
    except nx.NetworkXUnbounded:
        # Negative cycle detected
        path = None
        cost = -np.inf
    print("start:", s, " end:", d, " path:", path, " cost:", cost)
None

start: 0  end: 1  path: None  cost: -inf
start: 0  end: 2  path: None  cost: -inf
start: 0  end: 3  path: None  cost: -inf
start: 0  end: 4  path: None  cost: -inf
start: 0  end: 5  path: None  cost: -inf
start: 0  end: 6  path: None  cost: -inf
start: 0  end: 7  path: None  cost: -inf
start: 0  end: 8  path: None  cost: -inf
start: 0  end: 9  path: None  cost: -inf
start: 0  end: 10  path: None  cost: -inf
start: 0  end: 11  path: None  cost: -inf
start: 0  end: 12  path: None  cost: -inf
start: 0  end: 13  path: None  cost: -inf
start: 0  end: 14  path: None  cost: -inf
start: 0  end: 15  path: None  cost: -inf
start: 0  end: 16  path: None  cost: -inf
start: 0  end: 17  path: None  cost: -inf
start: 0  end: 18  path: None  cost: -inf
start: 0  end: 19  path: None  cost: -inf
start: 1  end: 2  path: None  cost: -inf
start: 1  end: 3  path: None  cost: -inf
start: 1  end: 4  path: None  cost: -inf
start: 1  end: 5  path: None  cost: -inf
start: 1  end: 6  path: None  cost: -inf
start:

In [ ]:

# this algorithm finds all possible paths from start to goal using a best-first search strategy but if the 
# problem size increases it's not efficient, I use it only for small problems in  order to verify 
# my bellman-ford implementation
def best_fit(graph: np.ndarray, start: int, goal: int) -> tuple[list[int], float]:
    
    size = graph.shape[0]
    
    # it keep track of all feasible paths
    feasible_paths=[]
    frontier=[]
    path_id = 0
    for i in range(size):
        if i != start and graph[start, i] != np.inf:
            frontier.append((path_id, i, graph[start, i])) # initial frontier 
            feasible_paths.append([start, i])# initial feasible paths
            path_id += 1

    goal_path=[]    

    while frontier:
        path_id, current_node, current_cost = frontier.pop(0)
        if current_node == goal:
            path= feasible_paths[path_id]
            if path not in [p[0] for p in goal_path]:
             goal_path.append((path, current_cost))
             
            continue
            

        for neighbor in range(size):
            weight = graph[current_node, neighbor]
            if weight != np.inf and neighbor not in feasible_paths[path_id]:
                current_path= feasible_paths[path_id]
                new_path=current_path + [neighbor]
                ## check if new path is already in feasible paths
                if new_path not in [p for p in feasible_paths]:
                    ##add new path
                    new_path_id= len(feasible_paths)
                    feasible_paths.append(new_path)

        
                    frontier.append((new_path_id, neighbor, current_cost + weight))

        # order of expansion: best-first (lowest cost first)
        frontier.sort(key=lambda x: x[1]) 
        
    if goal_path:
         # return the path with the lowest cost
         return min(goal_path, key=lambda x:  x[1])
       
    return None, np.inf


s, d = 1, 2
path, cost = best_fit(problem, s, d)
print( "start:", s, " end:", d, " path:", path, " cost:", cost)

start: 1  end: 2  path: [1, 8, 7, 14, 11, 3, 12, 0, 10, 2]  cost: -45850.0


In [84]:
## Bellman-Ford implementation, denied to negative cycles
def distances_bellman_ford(problem: np.ndarray, start: int) -> tuple[list[int], list[float]]:
    size = problem.shape[0]
    distances = [np.inf] * size
    predecessors = [None] * size
    distances[start] = 0

    for _ in range(size - 1):
        for u in range(size):
            for v in range(size):
                #for each edge extract the weight
                weight = problem[u, v]
                if weight != np.inf and distances[u] + weight < distances[v] :
                    #if the distance from source to u plus w is less than the current distance from source to v, then update the distance of v
                    
                    # update only if it does not create a negative cycle
                    visited_nodes=[v]
                    is_negative_cycle= False
                    current_node=u
                    for _ in range(size):
                        current_node= predecessors[current_node]
                        if current_node is None:
                            break
                        if current_node in visited_nodes:
                            is_negative_cycle= True
                            break
                        visited_nodes.append(current_node)
                         

                    if not is_negative_cycle:
                        predecessors[v] = u
                        distances[v] = distances[u] + weight

    return predecessors, distances
   


def shorthest_path_bellman_ford(predecessors, distances, end: int) -> tuple[list[int], float]:
    path = []
    current_node = end 
    while current_node is not None:
        path.insert(0, current_node)
        current_node = predecessors[current_node]
    return path, distances[end]


predecessors, distances = distances_bellman_ford(problem, 1)
path, cost = shorthest_path_bellman_ford(predecessors, distances, 2)
print(path, cost)

[1, 8, 19, 0, 10, 2] -16676.0


In [68]:
def path_cost(path, problem):
    total_cost=0
    for i in range(len(path)-1):
        u= path[i]
        v= path[i+1]
        total_cost += problem[u,v]
    return total_cost

path_cost(path, problem)

np.float64(-170611.0)

In [ ]:

for s, d in combinations(range(problem.shape[0]), 2):
    predecessors, distances = distances_bellman_ford(problem, s)
    path, cost = shorthest_path_bellman_ford(predecessors, distances, d)
   # print("start:", s, " end:", d, " cost:", cost, "\t path:", path  )
    if problem.shape[0] <= 20:
    # confronto con best fit
        path_2, cost_2 = best_fit(problem, s, d)
        if cost_2 != cost:
            print("Mismatch detected!")
            print(f"Nodes: {s} -> {d}")
            print(f"Bellman-Ford path: {path}, cost: {cost}")
            print(f"Best-fit path: {path_2}, cost: {cost_2}")
None

Mismatch detected!
Nodes: 0 -> 2
Bellman-Ford path: [0, 10, 2], cost: -10354.0
Best-fit path: [0, 10, 9, 7, 14, 11, 3, 12, 2], cost: -31358.0
Mismatch detected!
Nodes: 0 -> 4
Bellman-Ford path: [0, 10, 2, 4], cost: -19373.0
Best-fit path: [0, 10, 9, 7, 14, 11, 3, 12, 2, 4], cost: -40377.0
Mismatch detected!
Nodes: 0 -> 6
Bellman-Ford path: [0, 10, 6], cost: -3372.0
Best-fit path: [0, 8, 7, 9, 15, 3, 12, 2, 4, 14, 11, 10, 6], cost: -13230.0
Mismatch detected!
Nodes: 0 -> 10
Bellman-Ford path: [0, 10], cost: -5916.0
Best-fit path: [0, 8, 7, 9, 15, 3, 12, 2, 4, 14, 11, 10], cost: -15774.0
Mismatch detected!
Nodes: 0 -> 14
Bellman-Ford path: [0, 10, 2, 4, 14], cost: -17321.0
Best-fit path: [0, 10, 9, 15, 18, 11, 3, 12, 2, 4, 14], cost: -17479.0
Mismatch detected!
Nodes: 1 -> 2
Bellman-Ford path: [1, 8, 19, 0, 10, 2], cost: -16676.0
Best-fit path: [1, 8, 7, 14, 11, 3, 12, 0, 10, 2], cost: -45850.0
Mismatch detected!
Nodes: 1 -> 3
Bellman-Ford path: [1, 8, 19, 0, 10, 2, 4, 14, 11, 3], cost: 